[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/certified-journeys/certified-journeys.github.io/blob/main/courses/polars-certified/notebooks/day-09-custom-functions-plugins-and-python-udfs.ipynb#scrollTo=e1f2a3b4)

---
# Day 9 · Custom Functions, Plugins and Python UDFs
**certified-journeys / polars-certified** &nbsp;|&nbsp; Advanced

> **Goal for today:** Know when and how to use Python UDFs in Polars, understand the performance cost, and always prefer expressions first.

---
## The decision table: how to extend Polars

Before writing any custom function, consult this table:

| Approach | When to use | Performance | Parallelism |
|---|---|---|---|
| **Polars expression** | Logic expressible with built-ins | 🟢 Best (Rust, SIMD) | Full |
| **`map_elements()`** | Row-level Python, external library call | 🔴 Worst (Python per row) | None |
| **`map_batches()`** | Series-level NumPy/custom logic | 🟡 Medium (Python per batch) | Partial |
| **Polars plugin (Rust)** | Reusable, performance-critical custom logic | 🟢 Best (Rust) | Full |

> **Golden rule:** UDFs break Polars parallelism and the lazy optimizer. Always try to express logic as Polars expressions first — they run in Rust.

In [ ]:
%pip install -q polars

---
## Step 1 · map_elements — row-level Python UDF

`map_elements()` applies a Python function to **each element** one at a time.
Always specify `return_dtype` — without it, Polars must infer the type from the first element,
which can cause bugs with nulls and is slower.

In [ ]:
import polars as pl
import numpy as np
import time

rng = np.random.default_rng(42)
N = 100_000

df = pl.DataFrame({
    "text":    [f"product_{i % 500}" for i in range(N)],
    "score":   rng.uniform(0.0, 100.0, N),
    "weight":  rng.uniform(0.1, 2.0, N),
    "bonus":   rng.uniform(0.0, 10.0, N),
})

# map_elements: Python function called once per row
# Use case: calling an external library (e.g. a custom scoring function)
def custom_score(s: float) -> float:
    """Imagine this calls an external library or complex business logic."""
    return round(s ** 0.5 * 10, 4)  # trivial here, but the pattern is what matters

result = df.with_columns(
    pl.col("score")
      .map_elements(custom_score, return_dtype=pl.Float64)  # return_dtype is REQUIRED
      .alias("custom_score")
)
print(result.head(5))

**What just happened?**
- `map_elements()` called `custom_score` exactly 100,000 times — once per row.
- **`return_dtype=pl.Float64` is required**: without it, Polars infers from the first result, which fails on nulls and adds overhead.
- The GIL is held during each Python call — this prevents Polars from parallelising across threads.
- **Use this pattern only when you genuinely need Python per element** (e.g., calling a geocoder, tokenizer, or REST API).

---
## Step 2 · Performance cost: map_elements vs native expression

The same computation expressed as a Polars expression runs in Rust — orders of magnitude faster.
This benchmark makes the cost of UDFs concrete.

In [ ]:
REPS = 3

def bench(fn, reps=REPS):
    fn()  # warm-up
    times = []
    for _ in range(reps):
        t0 = time.perf_counter()
        fn()
        times.append((time.perf_counter() - t0) * 1000)
    return min(times)

# 1. UDF approach: map_elements (Python per row)
def with_udf():
    return df.with_columns(
        pl.col("score")
          .map_elements(lambda s: round(s ** 0.5 * 10, 4), return_dtype=pl.Float64)
          .alias("result")
    )

# 2. Expression approach: the exact same math, pure Polars
def with_expr():
    return df.with_columns(
        (pl.col("score").sqrt() * 10)  # same math, runs in Rust
          .alias("result")
    )

t_udf  = bench(with_udf)
t_expr = bench(with_expr)

print(f"map_elements (UDF):      {t_udf:.1f} ms")
print(f"Polars expression:       {t_expr:.1f} ms")
print(f"Expression is {t_udf/t_expr:.0f}x faster than UDF")

**What just happened?**
- The expression is typically **20–100x faster** than the equivalent UDF — the difference is Python call overhead multiplied by 100,000 rows.
- **Before writing a UDF, always ask:** can this be expressed with `pl.col().sqrt()`, `.str.contains()`, `.dt.month()`, or any other built-in expression?
- The performance gap grows with dataset size — at 10M rows, the UDF may take minutes; the expression takes milliseconds.
- **The lazy optimizer cannot see inside a UDF** and cannot reorder, fuse, or push it down.

---
## Step 3 · map_batches — Series-level functions

`map_batches()` receives an entire `Series` and must return a `Series`.
Much faster than `map_elements()` because Python is called **once**, not N times.

In [ ]:
import math

# map_batches: receives the WHOLE Series, returns a Series
# Use when you need NumPy/SciPy on the whole column at once
def numpy_sigmoid(series: pl.Series) -> pl.Series:
    """Apply sigmoid using NumPy — operates on the entire array at once."""
    arr = series.to_numpy()                        # zero-copy when possible
    result = 1.0 / (1.0 + np.exp(-arr / 50.0))    # sigmoid scaled to [0,100] range
    return pl.Series(result)                        # back to Polars Series

result_batched = df.with_columns(
    pl.col("score")
      .map_batches(numpy_sigmoid, return_dtype=pl.Float64)
      .alias("sigmoid_score")
)
print(result_batched.select(["score", "sigmoid_score"]).head(5))

# Benchmark: map_batches vs map_elements for the same transform
def with_map_elements():
    return df.with_columns(
        pl.col("score")
          .map_elements(
              lambda x: 1.0 / (1.0 + math.exp(-x / 50.0)),
              return_dtype=pl.Float64
          ).alias("r")
    )

def with_map_batches():
    return df.with_columns(
        pl.col("score")
          .map_batches(numpy_sigmoid, return_dtype=pl.Float64)
          .alias("r")
    )

t_elem   = bench(with_map_elements)
t_batch  = bench(with_map_batches)
print(f"\nmap_elements: {t_elem:.1f} ms")
print(f"map_batches:  {t_batch:.1f} ms  ({t_elem/t_batch:.1f}x faster)")

**What just happened?**
- `map_batches()` calls Python **once** with the entire column as a NumPy array — far fewer Python/C crossings.
- The NumPy vectorized operation `np.exp(-arr)` uses SIMD internally, approaching Polars expression speed.
- **Use `map_batches` when**: you need a NumPy/SciPy function not available in Polars, or you're applying a whole-array transform.
- **Use `map_elements` when**: you genuinely need the Python value of each element (e.g., calling an HTTP API per row — though you should batch those too).

---
## Step 4 · pl.fold and pl.reduce for custom horizontal aggregations

`pl.fold()` applies a function **across columns** (horizontally), accumulating a result.
Use it for weighted sums, custom scoring, or combining N columns into one.

In [ ]:
# Dataset: 3 score columns we want to combine into a weighted sum
scores_df = pl.DataFrame({
    "score_a": [80.0, 70.0, 90.0, 60.0],
    "score_b": [75.0, 85.0, 65.0, 95.0],
    "score_c": [90.0, 80.0, 70.0, 85.0],
})
weights = {"score_a": 0.5, "score_b": 0.3, "score_c": 0.2}  # must sum to 1.0

# pl.fold: start with an accumulator, apply a binary function column by column
weighted_sum = pl.fold(
    acc=pl.lit(0.0),                              # accumulator starts at 0
    function=lambda acc, col: acc + col,          # binary combine function
    exprs=[
        pl.col(name) * weight
        for name, weight in weights.items()
    ],
)

result = scores_df.with_columns(weighted_sum.alias("weighted_score"))
print("Weighted score via pl.fold:")
print(result)

# pl.reduce: like fold but uses the first expression as the initial accumulator
# Useful when all columns have the same weight (simple sum)
col_sum = pl.reduce(
    function=lambda acc, col: acc + col,
    exprs=[pl.col("score_a"), pl.col("score_b"), pl.col("score_c")],
)
print("\nSimple sum via pl.reduce:")
print(scores_df.with_columns(col_sum.alias("total_score")))

**What just happened?**
- `pl.fold()` iterates over the list of expressions, applying the binary function cumulatively — like Python's `functools.reduce` but over columns, not rows.
- **This is a pure expression** — no Python called per row, the optimizer can see it, parallelism is maintained.
- `pl.reduce()` is the same but uses the first expression as the starting accumulator — simpler syntax for equal-weight combinations.
- **Alternative for simple cases:** `pl.sum_horizontal([col_a, col_b, col_c])` is even more readable when all columns should be summed with weight 1.

---
## Step 5 · Struct operations: pack and apply

You can pack multiple columns into a `Struct`, then apply a UDF to the whole struct at once.
This is more efficient than calling `map_elements` on each column separately.

In [ ]:
# Pack score + weight into a Struct, then compute adjusted score in one UDF call
def adjusted_score(struct_val: dict) -> float:
    """Receives a dict with field names matching the struct schema."""
    # This simulates logic that needs BOTH score and weight simultaneously
    raw   = struct_val["score"]
    wt    = struct_val["weight"]
    bonus = struct_val["bonus"]
    return round(raw * wt + bonus, 4)

result_struct = df.with_columns(
    pl.struct(["score", "weight", "bonus"])      # pack 3 cols into one Struct col
      .map_elements(adjusted_score, return_dtype=pl.Float64)
      .alias("adjusted")
)
print("Struct-based UDF result:")
print(result_struct.select(["score", "weight", "bonus", "adjusted"]).head(5))

# Real-world case: when map_elements is genuinely necessary
# Example: calling a geocoder, NLP sentiment model, or regex library per row
import re

def extract_product_number(text: str) -> int:
    """Extract integer from strings like 'product_42' — regex per row."""
    m = re.search(r'(\d+)', text)
    return int(m.group(1)) if m else -1

# NOTE: for this specific case, a Polars expression is BETTER:
# pl.col("text").str.extract(r'(\d+)', 1).cast(pl.Int64)
# But map_elements is shown here to demonstrate the pattern for cases
# where no Polars built-in exists (e.g., spaCy NER, custom geocoder)
result_regex = df.with_columns(
    pl.col("text")
      .map_elements(extract_product_number, return_dtype=pl.Int64)
      .alias("product_num")
)
print("\nRegex UDF result (prefer pl.str.extract in production):")
print(result_regex.select(["text", "product_num"]).head(5))

**What just happened?**
- `pl.struct([...])` bundles multiple columns into one `Struct` column, then `map_elements` is called once per row with a `dict` containing all field values.
- This is cleaner than separate `map_elements` calls and avoids accidental column-count mismatch bugs.
- **The regex example** shows when a UDF is genuinely needed: calling `spaCy`, `geopy`, a custom ML model, or any library that has no Polars built-in equivalent.
- **Always check**: `pl.col().str.extract()`, `pl.col().str.contains()`, `pl.col().str.split()` before reaching for regex UDFs.

---
## Step 6 · Polars plugins (Rust extensions) — concept

For custom logic that must be reused, shared, and fast, you can write a **Polars plugin** in Rust.
This runs at full Polars speed with parallelism and optimizer integration.

**You do not need this for most tasks** — expressions and `map_batches` cover 95% of use cases.
Plugins are for library authors and performance-critical production systems.

```python
# Example: a hypothetical custom plugin named 'my_plugin'
# pip install my-polars-plugin  (published to PyPI)

import my_plugin  # registers custom expressions into Polars namespace

result = df.with_columns(
    pl.col("text").my_plugin.custom_tokenize()  # runs in Rust, full parallelism
)
```

**To build a plugin (requires Rust toolchain):**
```bash
pip install maturin polars
# maturin new my_plugin --bindings pyo3
# Implement your logic using polars-core crate
# maturin develop --release
```

Real examples to study:
- [polars-xdt](https://github.com/pola-rs/polars-xdt) — extended datetime operations
- [polars-distance](https://github.com/ion-elgreco/polars-distance) — string distance metrics

**When to write a plugin:** you've tried expressions and `map_batches`, the bottleneck is still the Python boundary, and the operation will be used across many pipelines.

---
## Challenge

Replace a `map_elements` call with a pure Polars expression and benchmark the speedup.

In [ ]:
# Given: a map_elements UDF that capitalises the first letter and appends ' Ltd'
companies = pl.DataFrame({
    "name": [f"company_{i}" for i in range(100_000)]
})

def with_udf():
    return companies.with_columns(
        pl.col("name")
          .map_elements(
              lambda s: s.replace("_", " ").title() + " Ltd",
              return_dtype=pl.String
          ).alias("formatted")
    )

# TODO: rewrite with_udf using only Polars string expressions:
# Hint: pl.col().str.replace(), pl.col().str.to_titlecase(), pl.col() + " Ltd"
def with_expression():
    return companies.with_columns(
        ...  # your pure Polars expression here
    )

# TODO: benchmark both
# t_udf  = bench(with_udf)
# t_expr = bench(with_expression)
# print(f"UDF:        {t_udf:.1f} ms")
# print(f"Expression: {t_expr:.1f} ms  ({t_udf/t_expr:.0f}x faster)")

---
## Day 9 recap

| Approach | Python calls | Parallelism | Optimizer-visible | When to use |
|---|---|---|---|---|
| Expression | 0 | Full | Yes | Always try first |
| `map_batches` | 1 per column | Partial | No | NumPy/SciPy whole-column ops |
| `map_elements` | N (one per row) | None | No | External library per row only |
| Plugin (Rust) | 0 | Full | Yes | Library authors, critical paths |

> **Tip:** UDFs break Polars parallelism. Always try to express logic as Polars expressions first — they run in Rust. The benchmark in Step 2 shows the cost clearly: a simple math expression is 20–100x faster than the equivalent `map_elements` call. When you genuinely need Python per-element (geocoders, NLP models, REST calls), batch your requests to minimise call overhead.

---
**What's next → Day 10:** Capstone — build a complete production analytics pipeline combining all concepts from Days 1–9.

Mark Day 9 complete in your [tracker](../index.html).